# 27d — Catching Negative Feature Extraction and Master Rebuild (corregido)

## Objetivo

Corregir la extracción de las tres X estructurales antes de reconstruir el dataset maestro.

La primera ejecución de `27d` se detuvo correctamente porque el extractor candidato **no reprodujo** los valores congelados de los dos sitios Catching `CONFIRMED_STRONG`.

Diferencias observadas:

```text
startbootstrap-3-col-portfolio
depth frozen = 8
depth new    = 10

startbootstrap-sb-admin-2
depth frozen = 15
depth new    = 17
```

y para `startbootstrap-sb-admin-2`:

```text
label ratio frozen = 0.842735...
label ratio new    = 0.856410...
```

El conteo de controles interactivos sí coincidió.

Por ello este notebook no asume una corrección manual. En su lugar:

1. carga los valores congelados;
2. ejecuta una búsqueda reproducible de variantes operacionales;
3. selecciona únicamente configuraciones que reproduzcan exactamente los tres features congelados;
4. si existe una configuración compatible, extrae X de los 12 repositorios negativos;
5. recupera Y desde `catching_targets.csv`;
6. reconstruye `semisupervised_proxy_master_dataset_v2.csv`;
7. vuelve a auditar el balance 0/1 y la posibilidad de evaluación agrupada.

No se crean negativos ni pseudo-etiquetas.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import itertools

try:
    from bs4 import BeautifulSoup, Tag
except ImportError as exc:
    raise ImportError(
        "Se requiere beautifulsoup4."
    ) from exc

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = PROJECT_DIR / "reports" / "tables"
PUBLIC_INTERIM_DIR = PROJECT_DIR / "data" / "public_v2" / "interim"

SEMISUP_DIR = PROJECT_DIR / "data" / "semisupervised"
SEMISUP_REPORTS_DIR = REPORTS_DIR / "semisupervised"
SEMISUP_INTERIM_DIR = SEMISUP_DIR / "interim"

SEMISUP_REPORTS_DIR.mkdir(parents=True, exist_ok=True)
SEMISUP_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

FEATURES = [
    "structure_depth",
    "interactive_control_count",
    "interactive_label_ratio"
]

print("PROJECT_DIR:", PROJECT_DIR)


PROJECT_DIR: /Users/uriel/Documents/machine_learning


## 2. Cargar artefactos

In [2]:
queue = pd.read_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_feature_extraction_queue.csv"
)

targets = pd.read_csv(
    PUBLIC_INTERIM_DIR
    / "catching_targets.csv"
)

frozen_site = pd.read_csv(
    PUBLIC_INTERIM_DIR
    / "catching_static_site_features.csv"
)

frozen_page_path = (
    PUBLIC_INTERIM_DIR
    / "catching_static_page_features.csv"
)

frozen_page = (
    pd.read_csv(frozen_page_path)
    if frozen_page_path.exists()
    else pd.DataFrame()
)

confirmed_registry = pd.read_csv(
    REPORTS_DIR
    / "catching_confirmed_repository_registry.csv"
)

current_master = pd.read_csv(
    SEMISUP_INTERIM_DIR
    / "semisupervised_proxy_master_dataset.csv"
)

print("Queue:", queue.shape)
print("Targets:", targets.shape)
print("Frozen site:", frozen_site.shape)
print("Frozen page:", frozen_page.shape)
print("Confirmed registry:", confirmed_registry.shape)
print("Current master:", current_master.shape)

display(frozen_site)


Queue: (12, 10)
Targets: (30, 8)
Frozen site: (2, 12)
Frozen page: (15, 14)
Confirmed registry: (2, 16)
Current master: (1002, 22)


,site,repo_full_name,static_page_count,static_mean_element_count,static_max_element_count,static_mean_dom_depth,static_max_dom_depth,static_total_interactive_controls,static_control_label_proxy_ratio,static_total_form_controls,static_native_validation_ratio,static_mean_class_reuse_ratio
0,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,1.0,98.000000,98.0,8.000000,8.0,23.0,0.739130,0.0,NaN,0.800000
1,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,14.0,232.357143,623.0,13.142857,15.0,585.0,0.842735,31.0,0.096774,0.737178


## 3. Descubrimiento de HTML

Se mantiene el mismo universo de archivos usado en la primera versión:

- `.html`
- `.htm`

y se excluyen:

- `.git`
- `node_modules`
- `vendor`


In [3]:
EXCLUDED_DIRS = {
    ".git",
    "node_modules",
    "vendor"
}


def discover_html_files(repo_path):

    repo_path = Path(repo_path)

    files = []

    for path in repo_path.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in {
            ".html",
            ".htm"
        }:
            continue

        if any(
            part in EXCLUDED_DIRS
            for part in path.parts
        ):
            continue

        files.append(path)

    return sorted(files)


## 4. Representación base de cada control

Para evitar volver a parsear HTML cientos de veces, primero se extraen propiedades elementales.

Después las distintas configuraciones de label proxy se calculan sobre estas propiedades.


In [4]:
def visible_text(tag):
    return " ".join(
        tag.stripped_strings
    ).strip()


def raw_tag_depth(tag):

    depth = 1
    parent = tag.parent

    while isinstance(parent, Tag):
        depth += 1
        parent = parent.parent

    return depth


def control_signature(control, soup):

    control_id = control.get("id")

    associated_label = False

    if control_id:

        label = soup.find(
            "label",
            attrs={
                "for": control_id
            }
        )

        associated_label = (
            label is not None
            and
            bool(
                visible_text(label)
            )
        )

    ancestor_label = control.find_parent(
        "label"
    )

    ancestor_label_text = (
        ancestor_label is not None
        and
        bool(
            visible_text(
                ancestor_label
            )
        )
    )

    input_type = (
        str(
            control.get(
                "type",
                "text"
            )
        ).lower()
        if control.name == "input"
        else ""
    )

    return {
        "has_text":
            bool(
                visible_text(
                    control
                )
            ),

        "has_aria_label":
            bool(
                str(
                    control.get(
                        "aria-label",
                        ""
                    )
                ).strip()
            ),

        "has_aria_labelledby":
            bool(
                str(
                    control.get(
                        "aria-labelledby",
                        ""
                    )
                ).strip()
            ),

        "has_title":
            bool(
                str(
                    control.get(
                        "title",
                        ""
                    )
                ).strip()
            ),

        "has_placeholder":
            bool(
                str(
                    control.get(
                        "placeholder",
                        ""
                    )
                ).strip()
            ),

        "has_button_value":
            (
                input_type
                in {
                    "button",
                    "submit",
                    "reset"
                }
                and
                bool(
                    str(
                        control.get(
                            "value",
                            ""
                        )
                    ).strip()
                )
            ),

        "has_associated_label":
            bool(
                associated_label
            ),

        "has_ancestor_label":
            bool(
                ancestor_label_text
            )
    }


def parse_html_base(html_path):

    try:

        text = Path(
            html_path
        ).read_text(
            encoding="utf-8",
            errors="ignore"
        )

        soup = BeautifulSoup(
            text,
            "html.parser"
        )

    except Exception as exc:

        return {
            "parse_ok": False,
            "error": repr(exc)
        }

    tags = soup.find_all(
        True
    )

    interactive = []

    for tag in tags:

        if (
            tag.name == "a"
            and
            tag.has_attr(
                "href"
            )
        ):

            interactive.append(
                tag
            )

        elif tag.name in {
            "button",
            "input",
            "select",
            "textarea"
        }:

            interactive.append(
                tag
            )

    signatures = [
        control_signature(
            control,
            soup
        )
        for control in interactive
    ]

    return {
        "parse_ok":
            True,

        "raw_max_depth":
            (
                max(
                    raw_tag_depth(
                        tag
                    )
                    for tag in tags
                )
                if tags
                else 0
            ),

        "interactive_controls":
            len(
                interactive
            ),

        "signatures":
            signatures,

        "error":
            ""
    }


## 5. Cargar repositorios congelados y parsearlos una sola vez

In [5]:
frozen_inputs = (
    confirmed_registry[
        [
            "site",
            "repo_full_name",
            "local_path"
        ]
    ]
    .drop_duplicates(
        subset=[
            "site"
        ]
    )
)

frozen_base = {}

for _, row in (
    frozen_inputs.iterrows()
):

    repo_path = Path(
        row[
            "local_path"
        ]
    )

    if not repo_path.is_absolute():

        repo_path = (
            PROJECT_DIR
            /
            repo_path
        )

    pages = []

    for html_file in discover_html_files(
        repo_path
    ):

        parsed = parse_html_base(
            html_file
        )

        pages.append({
            "relative_path":
                str(
                    html_file.relative_to(
                        repo_path
                    )
                ),
            **parsed
        })

    frozen_base[
        row[
            "site"
        ]
    ] = {
        "repo_full_name":
            row[
                "repo_full_name"
            ],
        "repo_path":
            repo_path,
        "pages":
            pages
    }

print(
    {
        site: len(
            info["pages"]
        )
        for site, info
        in frozen_base.items()
    }
)


{'startbootstrap-3-col-portfolio': 1, 'startbootstrap-sb-admin-2': 14}


## 6. Espacio de configuraciones candidatas

La primera ejecución mostró que:

- el depth nuevo fue exactamente `+2` en ambos sitios;
- el conteo interactivo fue exacto;
- la discrepancia restante estuvo en la definición de *label proxy*.

Se exploran reproduciblemente:

- offsets de profundidad `0, 1, 2, 3`;
- uso/no uso de:
  - `aria-label`;
  - `aria-labelledby`;
  - `title`;
  - `placeholder`;
  - value en botón/submit/reset;
  - `<label for>`;
  - `<label>` ancestro.

El texto visible siempre se conserva porque el sitio portfolio ya coincidía con esa semántica.


In [6]:
LABEL_FLAGS = [
    "has_aria_label",
    "has_aria_labelledby",
    "has_title",
    "has_placeholder",
    "has_button_value",
    "has_associated_label",
    "has_ancestor_label"
]


def signature_is_labeled(
    signature,
    enabled_flags
):

    if signature[
        "has_text"
    ]:
        return True

    return any(
        signature[
            flag
        ]
        for flag in enabled_flags
    )


def evaluate_config_on_site(
    site_info,
    depth_offset,
    enabled_flags
):

    valid_pages = [
        page
        for page in site_info[
            "pages"
        ]
        if page[
            "parse_ok"
        ]
    ]

    total_interactive = sum(
        page[
            "interactive_controls"
        ]
        for page in valid_pages
    )

    total_labeled = 0

    depth_values = []

    for page in valid_pages:

        depth_values.append(
            max(
                0,
                page[
                    "raw_max_depth"
                ]
                -
                depth_offset
            )
        )

        for signature in page[
            "signatures"
        ]:

            total_labeled += int(
                signature_is_labeled(
                    signature,
                    enabled_flags
                )
            )

    ratio = (
        total_labeled
        /
        total_interactive
        if total_interactive > 0
        else np.nan
    )

    return {
        "static_max_dom_depth":
            max(
                depth_values
            )
            if depth_values
            else np.nan,

        "static_total_interactive_controls":
            int(
                total_interactive
            ),

        "static_control_label_proxy_ratio":
            ratio,

        "labeled_interactive_controls":
            int(
                total_labeled
            )
    }


## 7. Calibrar configuraciones contra los valores congelados

In [7]:
frozen_lookup = (
    frozen_site
    .set_index(
        "site"
    )
)

candidate_records = []

for depth_offset in [
    0,
    1,
    2,
    3
]:

    for bits in itertools.product(
        [
            False,
            True
        ],
        repeat=len(
            LABEL_FLAGS
        )
    ):

        enabled_flags = [
            flag
            for flag, enabled
            in zip(
                LABEL_FLAGS,
                bits
            )
            if enabled
        ]

        site_results = {}

        all_match = True

        for site, site_info in (
            frozen_base.items()
        ):

            result = evaluate_config_on_site(
                site_info,
                depth_offset,
                enabled_flags
            )

            site_results[
                site
            ] = result

            frozen_row = frozen_lookup.loc[
                site
            ]

            depth_match = (
                float(
                    result[
                        "static_max_dom_depth"
                    ]
                )
                ==
                float(
                    frozen_row[
                        "static_max_dom_depth"
                    ]
                )
            )

            interactive_match = (
                float(
                    result[
                        "static_total_interactive_controls"
                    ]
                )
                ==
                float(
                    frozen_row[
                        "static_total_interactive_controls"
                    ]
                )
            )

            ratio_match = np.isclose(
                float(
                    result[
                        "static_control_label_proxy_ratio"
                    ]
                ),
                float(
                    frozen_row[
                        "static_control_label_proxy_ratio"
                    ]
                ),
                atol=1e-12,
                rtol=1e-12
            )

            all_match &= (
                depth_match
                and
                interactive_match
                and
                ratio_match
            )

        candidate_records.append({
            "depth_offset":
                depth_offset,

            "enabled_flags":
                " | ".join(
                    enabled_flags
                ),

            "n_enabled_flags":
                len(
                    enabled_flags
                ),

            "all_frozen_sites_match":
                all_match
        })

calibration_grid = pd.DataFrame(
    candidate_records
)

exact_configs = calibration_grid[
    calibration_grid[
        "all_frozen_sites_match"
    ]
].copy()

print(
    "Configuraciones evaluadas:",
    len(
        calibration_grid
    )
)

print(
    "Configuraciones exactas:",
    len(
        exact_configs
    )
)

display(
    exact_configs
)


Configuraciones evaluadas: 512
Configuraciones exactas: 16


,depth_offset,enabled_flags,n_enabled_flags,all_frozen_sites_match
322,2,has_aria_label | has_associated_label,2,True
323,2,has_aria_label | has_associated_label | has_ancestor_label,3,True
326,2,has_aria_label | has_button_value | has_associated_label,3,True
327,2,has_aria_label | has_button_value | has_associated_label | has_ancestor_label,4,True
338,2,has_aria_label | has_title | has_associated_label,3,True
339,2,has_aria_label | has_title | has_associated_label | has_ancestor_label,4,True
342,2,has_aria_label | has_title | has_button_value | has_associated_label,4,True
343,2,has_aria_label | has_title | has_button_value | has_associated_label | has_ancestor_label,5,True
354,2,has_aria_label | has_aria_labelledby | has_associated_label,3,True
355,2,has_aria_label | has_aria_labelledby | has_associated_label | has_ancestor_label,4,True


## 8. Selección reproducible de configuración

Si hay una sola configuración exacta, se usa.

Si hay varias configuraciones exactas, se aplica una regla de parsimonia:

1. menor número de flags;
2. preferencia por atributos semánticos/accessibility sobre `title`;
3. orden lexicográfico como desempate completamente reproducible.

La selección se registra explícitamente para no ocultar la ambigüedad de identificación.


In [8]:
flag_preference_weight = {
    "has_aria_label": 1,
    "has_aria_labelledby": 2,
    "has_associated_label": 3,
    "has_ancestor_label": 4,
    "has_placeholder": 5,
    "has_button_value": 6,
    "has_title": 7
}


def config_preference_score(
    flag_text
):

    flags = [
        value.strip()
        for value in str(
            flag_text
        ).split("|")
        if value.strip()
    ]

    return sum(
        flag_preference_weight.get(
            flag,
            100
        )
        for flag in flags
    )


if exact_configs.empty:

    selected_config = None

    calibration_status = (
        "NO_EXACT_CONFIGURATION"
    )

else:

    exact_configs[
        "preference_score"
    ] = exact_configs[
        "enabled_flags"
    ].map(
        config_preference_score
    )

    exact_configs = exact_configs.sort_values(
        [
            "n_enabled_flags",
            "preference_score",
            "enabled_flags"
        ]
    ).reset_index(
        drop=True
    )

    selected_config = exact_configs.iloc[
        0
    ]

    calibration_status = (
        "UNIQUE_EXACT_CONFIGURATION"
        if len(
            exact_configs
        )
        ==
        1
        else
        "MULTIPLE_EXACT_CONFIGURATIONS_PARSimONY_SELECTED"
    )

print(
    "Calibration status:",
    calibration_status
)

if selected_config is not None:

    print(
        "Selected depth_offset:",
        int(
            selected_config[
                "depth_offset"
            ]
        )
    )

    print(
        "Selected flags:",
        selected_config[
            "enabled_flags"
        ]
    )

display(
    exact_configs
)


Calibration status: MULTIPLE_EXACT_CONFIGURATIONS_PARSimONY_SELECTED
Selected depth_offset: 2
Selected flags: has_aria_label | has_associated_label


,depth_offset,enabled_flags,n_enabled_flags,all_frozen_sites_match,preference_score
0,2,has_aria_label | has_associated_label,2,True,4
1,2,has_aria_label | has_aria_labelledby | has_associated_label,3,True,6
2,2,has_aria_label | has_associated_label | has_ancestor_label,3,True,8
3,2,has_aria_label | has_button_value | has_associated_label,3,True,10
4,2,has_aria_label | has_title | has_associated_label,3,True,11
5,2,has_aria_label | has_aria_labelledby | has_associated_label | has_ancestor_label,4,True,10
6,2,has_aria_label | has_aria_labelledby | has_button_value | has_associated_label,4,True,12
7,2,has_aria_label | has_aria_labelledby | has_title | has_associated_label,4,True,13
8,2,has_aria_label | has_button_value | has_associated_label | has_ancestor_label,4,True,14
9,2,has_aria_label | has_title | has_associated_label | has_ancestor_label,4,True,15


## 9. Regresión final del extractor seleccionado

In [9]:
if selected_config is None:

    selected_depth_offset = None
    selected_flags = []

    regression = pd.DataFrame()

    extractor_validated = False

else:

    selected_depth_offset = int(
        selected_config[
            "depth_offset"
        ]
    )

    selected_flags = [
        value.strip()
        for value in str(
            selected_config[
                "enabled_flags"
            ]
        ).split("|")
        if value.strip()
    ]

    regression_records = []

    for site, site_info in (
        frozen_base.items()
    ):

        result = evaluate_config_on_site(
            site_info,
            selected_depth_offset,
            selected_flags
        )

        frozen_row = frozen_lookup.loc[
            site
        ]

        regression_records.append({
            "site":
                site,

            "depth_frozen":
                frozen_row[
                    "static_max_dom_depth"
                ],

            "depth_new":
                result[
                    "static_max_dom_depth"
                ],

            "interactive_frozen":
                frozen_row[
                    "static_total_interactive_controls"
                ],

            "interactive_new":
                result[
                    "static_total_interactive_controls"
                ],

            "label_ratio_frozen":
                frozen_row[
                    "static_control_label_proxy_ratio"
                ],

            "label_ratio_new":
                result[
                    "static_control_label_proxy_ratio"
                ],

            "depth_match":
                (
                    float(
                        frozen_row[
                            "static_max_dom_depth"
                        ]
                    )
                    ==
                    float(
                        result[
                            "static_max_dom_depth"
                        ]
                    )
                ),

            "interactive_match":
                (
                    float(
                        frozen_row[
                            "static_total_interactive_controls"
                        ]
                    )
                    ==
                    float(
                        result[
                            "static_total_interactive_controls"
                        ]
                    )
                ),

            "label_ratio_match":
                np.isclose(
                    float(
                        frozen_row[
                            "static_control_label_proxy_ratio"
                        ]
                    ),
                    float(
                        result[
                            "static_control_label_proxy_ratio"
                        ]
                    ),
                    atol=1e-12,
                    rtol=1e-12
                )
        })

    regression = pd.DataFrame(
        regression_records
    )

    regression[
        "all_three_match"
    ] = regression[
        [
            "depth_match",
            "interactive_match",
            "label_ratio_match"
        ]
    ].all(
        axis=1
    )

    extractor_validated = bool(
        regression[
            "all_three_match"
        ].all()
    )

display(
    regression
)

print(
    "EXTRACTOR_VALIDATED:",
    extractor_validated
)


,site,depth_frozen,depth_new,interactive_frozen,interactive_new,label_ratio_frozen,label_ratio_new,depth_match,interactive_match,label_ratio_match,all_three_match
0,startbootstrap-3-col-portfolio,8.0,8,23.0,23,0.739130,0.739130,True,True,True,True
1,startbootstrap-sb-admin-2,15.0,15,585.0,585,0.842735,0.842735,True,True,True,True


EXTRACTOR_VALIDATED: True


## 10. Extractor definitivo parametrizado

Solo se utiliza si `EXTRACTOR_VALIDATED=True`.


In [10]:
def extract_site_features(
    site,
    repo_full_name,
    repo_path,
    depth_offset,
    enabled_flags
):

    repo_path = Path(
        repo_path
    )

    page_records = []

    for html_file in discover_html_files(
        repo_path
    ):

        parsed = parse_html_base(
            html_file
        )

        if not parsed[
            "parse_ok"
        ]:

            page_records.append({
                "site":
                    site,
                "repo_full_name":
                    repo_full_name,
                "relative_path":
                    str(
                        html_file.relative_to(
                            repo_path
                        )
                    ),
                "parse_ok":
                    False,
                "dom_depth":
                    np.nan,
                "interactive_controls":
                    np.nan,
                "labeled_interactive_controls":
                    np.nan,
                "control_label_proxy_ratio":
                    np.nan,
                "error":
                    parsed[
                        "error"
                    ]
            })

            continue

        labeled = sum(
            signature_is_labeled(
                signature,
                enabled_flags
            )
            for signature in parsed[
                "signatures"
            ]
        )

        interactive = parsed[
            "interactive_controls"
        ]

        page_records.append({
            "site":
                site,
            "repo_full_name":
                repo_full_name,
            "relative_path":
                str(
                    html_file.relative_to(
                        repo_path
                    )
                ),
            "parse_ok":
                True,
            "dom_depth":
                max(
                    0,
                    parsed[
                        "raw_max_depth"
                    ]
                    -
                    depth_offset
                ),
            "interactive_controls":
                interactive,
            "labeled_interactive_controls":
                int(
                    labeled
                ),
            "control_label_proxy_ratio":
                (
                    labeled
                    /
                    interactive
                    if interactive > 0
                    else np.nan
                ),
            "error":
                ""
        })

    pages = pd.DataFrame(
        page_records
    )

    if pages.empty:

        return (
            {
                "site":
                    site,
                "repo_full_name":
                    repo_full_name,
                "static_page_count":
                    0,
                "static_max_dom_depth":
                    np.nan,
                "static_total_interactive_controls":
                    np.nan,
                "static_control_label_proxy_ratio":
                    np.nan,
                "extract_ok":
                    False,
                "extract_reason":
                    "NO_HTML_FILES"
            },
            pages
        )

    valid = pages[
        pages[
            "parse_ok"
        ]
    ].copy()

    if valid.empty:

        return (
            {
                "site":
                    site,
                "repo_full_name":
                    repo_full_name,
                "static_page_count":
                    len(
                        pages
                    ),
                "static_max_dom_depth":
                    np.nan,
                "static_total_interactive_controls":
                    np.nan,
                "static_control_label_proxy_ratio":
                    np.nan,
                "extract_ok":
                    False,
                "extract_reason":
                    "NO_PARSEABLE_HTML"
            },
            pages
        )

    total_interactive = int(
        valid[
            "interactive_controls"
        ].sum()
    )

    total_labeled = int(
        valid[
            "labeled_interactive_controls"
        ].sum()
    )

    return (
        {
            "site":
                site,
            "repo_full_name":
                repo_full_name,
            "static_page_count":
                len(
                    valid
                ),
            "static_max_dom_depth":
                int(
                    valid[
                        "dom_depth"
                    ].max()
                ),
            "static_total_interactive_controls":
                total_interactive,
            "static_control_label_proxy_ratio":
                (
                    total_labeled
                    /
                    total_interactive
                    if total_interactive > 0
                    else np.nan
                ),
            "extract_ok":
                True,
            "extract_reason":
                ""
        },
        pages
    )


## 11. Extraer X de los 12 repositorios negativos

In [11]:
negative_site_records = []
negative_page_frames = []

if extractor_validated:

    for _, row in (
        queue.iterrows()
    ):

        local_path = Path(
            row[
                "local_path"
            ]
        )

        if not local_path.is_absolute():

            local_path = (
                PROJECT_DIR
                /
                local_path
            )

        site_row, page_df = (
            extract_site_features(
                site=row[
                    "site_or_instance"
                ],
                repo_full_name=row[
                    "repository"
                ],
                repo_path=local_path,
                depth_offset=
                    selected_depth_offset,
                enabled_flags=
                    selected_flags
            )
        )

        site_row[
            "source_match_level"
        ] = row[
            "evidence_tier"
        ]

        site_row[
            "strict_source_match"
        ] = row[
            "strict_source_match"
        ]

        site_row[
            "extended_source_match"
        ] = row[
            "extended_source_match"
        ]

        negative_site_records.append(
            site_row
        )

        if not page_df.empty:

            page_df[
                "source_match_level"
            ] = row[
                "evidence_tier"
            ]

            negative_page_frames.append(
                page_df
            )

negative_site_features = pd.DataFrame(
    negative_site_records
)

negative_page_features = (
    pd.concat(
        negative_page_frames,
        ignore_index=True
    )
    if negative_page_frames
    else pd.DataFrame()
)

print(
    "Negative site features:",
    negative_site_features.shape
)

display(
    negative_site_features
)


Negative site features: (12, 11)


,site,repo_full_name,static_page_count,static_max_dom_depth,static_total_interactive_controls,static_control_label_proxy_ratio,extract_ok,extract_reason,source_match_level,strict_source_match,extended_source_match
0,beginner-html-site-styled,mdn/beginner-html-site-styled,1,3.0,1.0,1.000000,True,,B_MODERATE_VALIDATED,False,True
1,conference-website-template,mikepierce/conference-website-template,5,6.0,41.0,0.878049,True,,B_MODERATE_VALIDATED,False,True
2,personal-website-template,The-WebOps-Club/personal-website-template,1,11.0,48.0,0.729167,True,,B_MODERATE_VALIDATED,False,True
3,react-saas-template,dunky11/react-saas-template,1,2.0,0.0,NaN,True,,B_MODERATE_VALIDATED,False,True
4,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,0,NaN,NaN,NaN,False,NO_HTML_FILES,C_EXACT_NAME_TECHNICAL,False,True
5,charity_responsive-template,website-templates/charity_responsive-template,1,7.0,25.0,0.840000,True,,C_EXACT_NAME_TECHNICAL,False,True
6,simple-gallery_responsive-template,website-templates/simple-gallery_responsive-template,3,15.0,38.0,0.973684,True,,C_EXACT_NAME_TECHNICAL,False,True
7,single-page-small-business-web-application,reza-azimifar/single-page-small-business-web-application,2,8.0,60.0,0.833333,True,,C_EXACT_NAME_TECHNICAL,False,True
8,startbootstrap-freelancer,StartBootstrap/startbootstrap-freelancer,1,11.0,29.0,0.862069,True,,C_EXACT_NAME_TECHNICAL,False,True
9,startbootstrap-landing-page,StartBootstrap/startbootstrap-landing-page,1,10.0,15.0,0.666667,True,,C_EXACT_NAME_TECHNICAL,False,True


## 12. Auditar X

In [12]:
if negative_site_features.empty:

    x_ready = False

    feature_audit = pd.DataFrame([
        {
            "check":
                "NEGATIVE_FEATURES_AVAILABLE",
            "passed":
                False,
            "detail":
                "No rows extracted."
        }
    ])

else:

    complete_x = (
        pd.to_numeric(
            negative_site_features[
                "static_max_dom_depth"
            ],
            errors="coerce"
        ).notna()
        &
        pd.to_numeric(
            negative_site_features[
                "static_total_interactive_controls"
            ],
            errors="coerce"
        ).notna()
        &
        pd.to_numeric(
            negative_site_features[
                "static_control_label_proxy_ratio"
            ],
            errors="coerce"
        ).notna()
    )

    negative_site_features[
        "complete_proxy_x"
    ] = complete_x

    x_ready = bool(
        extractor_validated
        and
        len(
            negative_site_features
        )
        ==
        len(
            queue
        )
        and
        complete_x.all()
    )

    feature_audit = pd.DataFrame([
        {
            "check":
                "EXTRACTOR_VALIDATED",
            "passed":
                extractor_validated,
            "detail":
                str(
                    extractor_validated
                )
        },
        {
            "check":
                "ALL_QUEUE_SITES_EXTRACTED",
            "passed":
                (
                    len(
                        negative_site_features
                    )
                    ==
                    len(
                        queue
                    )
                ),
            "detail":
                (
                    f"{len(negative_site_features)} / "
                    f"{len(queue)}"
                )
        },
        {
            "check":
                "ALL_PROXY_X_COMPLETE",
            "passed":
                bool(
                    complete_x.all()
                ),
            "detail":
                (
                    f"{int(complete_x.sum())} / "
                    f"{len(complete_x)}"
                )
        }
    ])

display(
    feature_audit
)

print(
    "X_READY:",
    x_ready
)


,check,passed,detail
0,EXTRACTOR_VALIDATED,True,True
1,ALL_QUEUE_SITES_EXTRACTED,True,12 / 12
2,ALL_PROXY_X_COMPLETE,False,10 / 12


X_READY: False


## 13. Recuperar vector completo Y desde Catching

In [13]:
def normalize_identity(value):

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower()
    )


targets = targets.copy()

targets[
    "__site_key"
] = targets[
    "site"
].map(
    normalize_identity
)

if x_ready:

    negative_site_features[
        "__site_key"
    ] = negative_site_features[
        "site"
    ].map(
        normalize_identity
    )

    target_columns = [
        "site",
        "__site_key",
        "source_dataset",
        "label_source",
        "label_level"
    ] + TARGETS

    target_columns = [
        col
        for col in target_columns
        if col in targets.columns
    ]

    negative_xy = (
        negative_site_features
        .merge(
            targets[
                target_columns
            ],
            on="__site_key",
            how="left",
            suffixes=(
                "_feature",
                "_target"
            ),
            validate="one_to_one"
        )
    )

else:

    negative_xy = pd.DataFrame()

display(
    negative_xy
)


""


## 14. Validar X + Y

In [14]:
if negative_xy.empty:

    xy_ready = False

    xy_audit = pd.DataFrame([
        {
            "check":
                "XY_READY",
            "passed":
                False,
            "detail":
                "No rows available."
        }
    ])

else:

    y_binary = pd.Series(
        True,
        index=negative_xy.index
    )

    for target in TARGETS:

        values = pd.to_numeric(
            negative_xy[
                target
            ],
            errors="coerce"
        )

        y_binary &= values.isin(
            [
                0,
                1
            ]
        )

    xy_ready = bool(
        len(
            negative_xy
        )
        ==
        len(
            queue
        )
        and
        y_binary.all()
    )

    xy_audit = pd.DataFrame([
        {
            "check":
                "ALL_QUEUE_ROWS_JOINED",
            "passed":
                (
                    len(
                        negative_xy
                    )
                    ==
                    len(
                        queue
                    )
                ),
            "detail":
                (
                    f"{len(negative_xy)} / "
                    f"{len(queue)}"
                )
        },
        {
            "check":
                "ALL_Y_BINARY",
            "passed":
                bool(
                    y_binary.all()
                ),
            "detail":
                (
                    f"{int(y_binary.sum())} / "
                    f"{len(y_binary)}"
                )
        }
    ])

display(
    xy_audit
)

print(
    "XY_READY:",
    xy_ready
)


,check,passed,detail
0,XY_READY,False,No rows available.


XY_READY: False


## 15. Construir filas extendidas

In [15]:
def normalize_id_part(value):

    value = str(
        value
    ).strip().lower()

    value = re.sub(
        r"[^a-z0-9]+",
        "_",
        value
    )

    return value.strip("_")


def master_id(
    source_dataset,
    source_instance_id
):

    return (
        normalize_id_part(
            source_dataset
        )
        +
        "__"
        +
        normalize_id_part(
            source_instance_id
        )
    )


extended_rows = []

if xy_ready:

    for _, row in (
        negative_xy.iterrows()
    ):

        site_value = row[
            "site_feature"
        ]

        extended_rows.append({
            "master_instance_id":
                master_id(
                    "Catching_UX_Flaws",
                    site_value
                ),
            "source_dataset":
                "Catching_UX_Flaws",
            "source_instance_id":
                str(
                    site_value
                ),
            "source_unit":
                "website",
            "feature_domain":
                "WEB_DOM",
            "feature_space":
                "EXPERIMENTAL_CONSTRUCT_PROXY",
            "structure_depth":
                pd.to_numeric(
                    row[
                        "static_max_dom_depth"
                    ],
                    errors="coerce"
                ),
            "interactive_control_count":
                pd.to_numeric(
                    row[
                        "static_total_interactive_controls"
                    ],
                    errors="coerce"
                ),
            "interactive_label_ratio":
                pd.to_numeric(
                    row[
                        "static_control_label_proxy_ratio"
                    ],
                    errors="coerce"
                ),
            "H1_violation":
                pd.to_numeric(
                    row[
                        "H1_violation"
                    ],
                    errors="coerce"
                ),
            "H4_violation":
                pd.to_numeric(
                    row[
                        "H4_violation"
                    ],
                    errors="coerce"
                ),
            "H5_violation":
                pd.to_numeric(
                    row[
                        "H5_violation"
                    ],
                    errors="coerce"
                ),
            "H6_violation":
                pd.to_numeric(
                    row[
                        "H6_violation"
                    ],
                    errors="coerce"
                ),
            "label_source":
                str(
                    row.get(
                        "label_source",
                        "Catching"
                    )
                ),
            "source_label_level":
                str(
                    row.get(
                        "label_level",
                        "LLM_DIRECT"
                    )
                ),
            "nielsen_label_level":
                str(
                    row.get(
                        "label_level",
                        "LLM_DIRECT"
                    )
                ),
            "is_pseudo_label":
                False,
            "group_id":
                (
                    "catching__"
                    +
                    normalize_id_part(
                        site_value
                    )
                ),
            "domain_shift_flag":
                "WEB_DOM",
            "complete_proxy_x":
                True,
            "has_any_observed_nielsen_y":
                True,
            "master_role":
                "OBSERVED_LABEL_SOURCE_EXTENDED",
            "source_match_level":
                row[
                    "source_match_level"
                ]
        })

extended_master_rows = pd.DataFrame(
    extended_rows
)

display(
    extended_master_rows
)


""


## 16. Reconstruir master v2 sin sobrescribir v1

In [16]:
master_v2_base = current_master.copy()

if "source_match_level" not in (
    master_v2_base.columns
):

    master_v2_base[
        "source_match_level"
    ] = np.where(
        master_v2_base[
            "source_dataset"
        ]
        ==
        "Catching_UX_Flaws",
        "A_CONFIRMED_STRONG",
        "NOT_APPLICABLE_UNLABELED"
    )

if extended_master_rows.empty:

    collisions = pd.DataFrame()

    master_v2 = (
        master_v2_base.copy()
    )

    master_v2_built = False

else:

    collision_ids = set(
        master_v2_base[
            "master_instance_id"
        ]
    ).intersection(
        set(
            extended_master_rows[
                "master_instance_id"
            ]
        )
    )

    collisions = (
        extended_master_rows[
            extended_master_rows[
                "master_instance_id"
            ].isin(
                collision_ids
            )
        ]
        .copy()
    )

    safe_extended = (
        extended_master_rows[
            ~extended_master_rows[
                "master_instance_id"
            ].isin(
                collision_ids
            )
        ]
        .copy()
    )

    master_v2 = pd.concat(
        [
            master_v2_base,
            safe_extended
        ],
        ignore_index=True,
        sort=False
    )

    master_v2_built = bool(
        xy_ready
        and
        len(
            safe_extended
        )
        >
        0
        and
        master_v2[
            "master_instance_id"
        ].duplicated().sum()
        ==
        0
    )

print(
    "Master v2 built:",
    master_v2_built
)

print(
    "Rows v1:",
    len(
        current_master
    )
)

print(
    "Rows added:",
    len(
        master_v2
    )
    -
    len(
        current_master
    )
)

print(
    "Rows v2:",
    len(
        master_v2
    )
)

print(
    "Collisions:",
    len(
        collisions
    )
)


Master v2 built: False
Rows v1: 1002
Rows added: 0
Rows v2: 1002
Collisions: 0


## 17. Balance etiquetado del master v2

In [17]:
labeled_catching = master_v2[
    master_v2[
        "source_dataset"
    ]
    ==
    "Catching_UX_Flaws"
].copy()

balance_records = []

for target in TARGETS:

    values = pd.to_numeric(
        labeled_catching[
            target
        ],
        errors="coerce"
    )

    balance_records.append({
        "target":
            target,
        "n_labeled":
            int(
                values.notna().sum()
            ),
        "n_0":
            int(
                (
                    values
                    ==
                    0
                ).sum()
            ),
        "n_1":
            int(
                (
                    values
                    ==
                    1
                ).sum()
            ),
        "n_classes":
            int(
                values.dropna().nunique()
            ),
        "both_classes_observed":
            (
                values.dropna().nunique()
                >=
                2
            )
    })

master_v2_balance = pd.DataFrame(
    balance_records
)

display(
    master_v2_balance
)


,target,n_labeled,n_0,n_1,n_classes,both_classes_observed
0,H1_violation,2,0,2,1,False
1,H4_violation,2,0,2,1,False
2,H5_violation,2,0,2,1,False
3,H6_violation,2,0,2,1,False


## 18. Soporte mínimo de grupos para evaluación

Para un holdout binario agrupado mínimo se pide:

```text
n_0 >= 2
n_1 >= 2
```

Esto solo indica que existe posibilidad lógica de dejar al menos un grupo por clase fuera del train.

No implica todavía que el tamaño sea estadísticamente suficiente.


In [18]:
target_readiness = (
    master_v2_balance.copy()
)

target_readiness[
    "grouped_holdout_class_support"
] = (
    (
        target_readiness[
            "n_0"
        ]
        >=
        2
    )
    &
    (
        target_readiness[
            "n_1"
        ]
        >=
        2
    )
)

target_readiness[
    "baseline_status"
] = np.select(
    [
        target_readiness[
            "grouped_holdout_class_support"
        ],
        target_readiness[
            "both_classes_observed"
        ]
    ],
    [
        "BASELINE_CANDIDATE",
        "TRAINABLE_BUT_GROUPED_HOLDOUT_BLOCKED"
    ],
    default="NOT_TRAINABLE"
)

display(
    target_readiness
)


,target,n_labeled,n_0,n_1,n_classes,both_classes_observed,grouped_holdout_class_support,baseline_status
0,H1_violation,2,0,2,1,False,False,NOT_TRAINABLE
1,H4_violation,2,0,2,1,False,False,NOT_TRAINABLE
2,H5_violation,2,0,2,1,False,False,NOT_TRAINABLE
3,H6_violation,2,0,2,1,False,False,NOT_TRAINABLE


## 19. Reconstruir membresía L/U v2

In [19]:
membership_records = []

for _, row in (
    master_v2.iterrows()
):

    complete_x = bool(
        pd.notna(
            row.get(
                "structure_depth"
            )
        )
        and
        pd.notna(
            row.get(
                "interactive_control_count"
            )
        )
        and
        pd.notna(
            row.get(
                "interactive_label_ratio"
            )
        )
    )

    for target in TARGETS:

        y = row[
            target
        ]

        if not complete_x:

            role = (
                "EXCLUDED_INCOMPLETE_X"
            )

        elif pd.isna(
            y
        ):

            role = "U"

        elif y in [
            0,
            1
        ]:

            role = "L"

        else:

            role = (
                "EXCLUDED_INVALID_Y"
            )

        membership_records.append({
            "master_instance_id":
                row[
                    "master_instance_id"
                ],
            "source_dataset":
                row[
                    "source_dataset"
                ],
            "source_instance_id":
                row[
                    "source_instance_id"
                ],
            "group_id":
                row[
                    "group_id"
                ],
            "source_match_level":
                row[
                    "source_match_level"
                ],
            "target":
                target,
            "observed_y":
                y,
            "ssl_role":
                role
        })

membership_v2 = pd.DataFrame(
    membership_records
)

split_summary_v2 = (
    membership_v2
    .groupby(
        [
            "target",
            "ssl_role"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

display(
    split_summary_v2
)


ssl_role,target,EXCLUDED_INCOMPLETE_X,L,U
0,H1_violation,2,2,998
1,H4_violation,2,2,998
2,H5_violation,2,2,998
3,H6_violation,2,2,998


## 20. Decisión final

In [20]:
full_baseline_ready = bool(
    master_v2_built
    and
    target_readiness[
        "grouped_holdout_class_support"
    ].all()
)

any_baseline_ready = bool(
    master_v2_built
    and
    target_readiness[
        "grouped_holdout_class_support"
    ].any()
)

if not extractor_validated:

    global_status = (
        "EXTRACTOR_STILL_NOT_REPRODUCIBLE"
    )

elif not master_v2_built:

    global_status = (
        "MASTER_V2_NOT_BUILT"
    )

elif full_baseline_ready:

    global_status = (
        "PROCEED_TO_28_FULL_BASELINE"
    )

elif any_baseline_ready:

    global_status = (
        "PROCEED_TO_28_PARTIAL_BASELINE"
    )

else:

    global_status = (
        "MASTER_V2_BUILT_BUT_BASELINE_VALIDATION_BLOCKED"
    )

decision = pd.DataFrame([
    {
        "calibration_status":
            calibration_status,
        "extractor_validated":
            extractor_validated,
        "master_v2_built":
            master_v2_built,
        "master_v2_rows":
            len(
                master_v2
            ),
        "labeled_catching_rows":
            len(
                labeled_catching
            ),
        "full_baseline_ready":
            full_baseline_ready,
        "any_baseline_ready":
            any_baseline_ready,
        "global_status":
            global_status
    }
])

display(
    decision
)


,calibration_status,extractor_validated,master_v2_built,master_v2_rows,labeled_catching_rows,full_baseline_ready,any_baseline_ready,global_status
0,MULTIPLE_EXACT_CONFIGURATIONS_PARSimONY_SELECTED,True,False,1002,2,False,False,MASTER_V2_NOT_BUILT


## 21. Guardar artefactos

In [21]:
calibration_grid.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_extractor_calibration_grid.csv",
    index=False
)

exact_configs.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_extractor_exact_configurations.csv",
    index=False
)

regression.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_extractor_regression_validation_v2.csv",
    index=False
)

negative_site_features.to_csv(
    SEMISUP_INTERIM_DIR
    / "catching_negative_static_site_features_v2.csv",
    index=False
)

negative_page_features.to_csv(
    SEMISUP_INTERIM_DIR
    / "catching_negative_static_page_features_v2.csv",
    index=False
)

negative_xy.to_csv(
    SEMISUP_INTERIM_DIR
    / "catching_negative_xy_extended_v2.csv",
    index=False
)

master_v2.to_csv(
    SEMISUP_INTERIM_DIR
    / "semisupervised_proxy_master_dataset_v2.csv",
    index=False
)

membership_v2.to_csv(
    SEMISUP_INTERIM_DIR
    / "semisupervised_target_membership_long_v2.csv",
    index=False
)

split_summary_v2.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_target_split_summary_v2.csv",
    index=False
)

master_v2_balance.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_label_balance.csv",
    index=False
)

target_readiness.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_target_readiness.csv",
    index=False
)

feature_audit.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_feature_extraction_audit_v2.csv",
    index=False
)

xy_audit.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_xy_join_audit_v2.csv",
    index=False
)

decision.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_decision.csv",
    index=False
)


In [22]:
expected_outputs = [
    SEMISUP_REPORTS_DIR
    / "catching_extractor_calibration_grid.csv",
    SEMISUP_REPORTS_DIR
    / "catching_extractor_exact_configurations.csv",
    SEMISUP_REPORTS_DIR
    / "catching_extractor_regression_validation_v2.csv",
    SEMISUP_INTERIM_DIR
    / "catching_negative_static_site_features_v2.csv",
    SEMISUP_INTERIM_DIR
    / "catching_negative_xy_extended_v2.csv",
    SEMISUP_INTERIM_DIR
    / "semisupervised_proxy_master_dataset_v2.csv",
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_label_balance.csv",
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_target_readiness.csv",
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_decision.csv"
]

for path in expected_outputs:

    print(
        path.relative_to(
            PROJECT_DIR
        ),
        "✅"
        if path.exists()
        else "❌"
    )


reports/tables/semisupervised/catching_extractor_calibration_grid.csv ✅
reports/tables/semisupervised/catching_extractor_exact_configurations.csv ✅
reports/tables/semisupervised/catching_extractor_regression_validation_v2.csv ✅
data/semisupervised/interim/catching_negative_static_site_features_v2.csv ✅
data/semisupervised/interim/catching_negative_xy_extended_v2.csv ✅
data/semisupervised/interim/semisupervised_proxy_master_dataset_v2.csv ✅
reports/tables/semisupervised/semisupervised_master_v2_label_balance.csv ✅
reports/tables/semisupervised/semisupervised_master_v2_target_readiness.csv ✅
reports/tables/semisupervised/semisupervised_master_v2_decision.csv ✅


## Punto de detención

No crear todavía `28_supervised_baseline.ipynb`.

El resultado que debe revisarse es:

```text
semisupervised_master_v2_decision.csv
```

y, en particular:

```text
extractor_validated
master_v2_built
full_baseline_ready
any_baseline_ready
global_status
```

Solo si el resultado permite baseline se construirá el notebook 28.

La calibración del extractor debe conservarse como evidencia de reproducibilidad metodológica, ya que documenta exactamente cómo se recuperó la definición operacional que produjo los features congelados.
